<a href="https://colab.research.google.com/github/SadakatSarin/Research-498R/blob/main/ProtoNet_Omniglot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Imports & Dataset Setup
This downloads the Omniglot dataset and groups images by character class.

In [1]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torchvision import datasets, transforms

# Set device and random seeds
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Download Omniglot (background = train, evaluation = test)
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor()
])

train_raw = datasets.Omniglot(root='./data', background=True, download=True, transform=transform)
test_raw  = datasets.Omniglot(root='./data', background=False, download=True, transform=transform)

# Group images by class label
def group_by_class(dataset):
    class_dict = {}
    for img, label in dataset:
        class_dict.setdefault(label, []).append(img)
    return class_dict

train_classes = group_by_class(train_raw)
test_classes  = group_by_class(test_raw)

print(f"Train classes: {len(train_classes)}, Test classes: {len(test_classes)}")

Using device: cpu


100%|██████████| 9.46M/9.46M [00:00<00:00, 115MB/s]
100%|██████████| 6.46M/6.46M [00:00<00:00, 82.8MB/s]


Train classes: 964, Test classes: 659


 Episodic Sampler (Creates N-way K-shot Tasks)Each episode randomly picks $N$ classes, $K$ support images per class, and $Q$ query images per class.   

In [2]:
def sample_episode(class_dict, n_way=5, k_shot=5, q_query=5):
    selected_classes = random.sample(list(class_dict.keys()), n_way)
    support_imgs, query_imgs = [], []

    for cls in selected_classes:
        imgs = random.sample(class_dict[cls], k_shot + q_query)
        support_imgs.extend(imgs[:k_shot])
        query_imgs.extend(imgs[k_shot:])

    support_x = torch.stack(support_imgs).to(device)  # [n_way * k_shot, 1, 28, 28]
    query_x   = torch.stack(query_imgs).to(device)    # [n_way * q_query, 1, 28, 28]

    # Labels for queries: [0, 0, ..., 1, 1, ..., n_way-1]
    query_y = torch.arange(n_way, device=device).repeat_interleave(q_query)
    return support_x, query_x, query_y

4-Block CNN Encoder & Prototypical Loss
This implements the exact 4-block encoder and squared Euclidean distance loss from the paper.

In [3]:
def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(),
        nn.MaxPool2d(2)
    )

class ProtoNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            conv_block(1, 64),
            conv_block(64, 64),
            conv_block(64, 64),
            conv_block(64, 64)
        )

    def forward(self, x):
        # Output shape for 28x28 input: [batch, 64, 1, 1] -> [batch, 64]
        return self.encoder(x).view(x.size(0), -1)

def prototypical_loss(model, support_x, query_x, query_y, n_way, k_shot):
    # 1. Embed support and query images
    z_support = model(support_x)  # [n_way * k_shot, 64]
    z_query   = model(query_x)    # [n_way * q_query, 64]

    # 2. Compute prototypes by averaging support vectors per class
    prototypes = z_support.view(n_way, k_shot, -1).mean(dim=1)  # [n_way, 64]

    # 3. Squared Euclidean distance between each query and each prototype
    dists = torch.cdist(z_query, prototypes) ** 2  # [n_way * q_query, n_way]

    # 4. Log-softmax over negative distances
    log_p_y = F.log_softmax(-dists, dim=1)
    loss = F.nll_loss(log_p_y, query_y)

    # Calculate accuracy
    preds = log_p_y.argmax(dim=1)
    acc = (preds == query_y).float().mean().item()
    return loss, acc

Train & Test on Unseen Classes
Following the paper's finding, we train with a higher "way" (20-way) than we test on (5-way).

In [4]:
model = ProtoNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Training: 20-way 5-shot (10 epochs x 100 episodes = 1,000 episodes)
for epoch in range(1, 11):
    model.train()
    total_loss, total_acc = 0, 0

    for _ in range(100):
        support_x, query_x, query_y = sample_episode(train_classes, n_way=20, k_shot=5, q_query=5)
        optimizer.zero_grad()
        loss, acc = prototypical_loss(model, support_x, query_x, query_y, n_way=20, k_shot=5)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += acc

    print(f"Epoch {epoch:02d} | Train Loss: {total_loss/100:.4f} | Train Acc (20-way): {total_acc:.2f}%")

# Evaluation on unseen test classes: 5-way 5-shot over 200 episodes
model.eval()
test_acc = 0
with torch.no_grad():
    for _ in range(200):
        support_x, query_x, query_y = sample_episode(test_classes, n_way=5, k_shot=5, q_query=5)
        _, acc = prototypical_loss(model, support_x, query_x, query_y, n_way=5, k_shot=5)
        test_acc += acc

print(f"\nTest Accuracy on Unseen Classes (5-way 5-shot): {test_acc/200 * 100:.2f}%")

Epoch 01 | Train Loss: 0.4536 | Train Acc (20-way): 86.77%
Epoch 02 | Train Loss: 0.1660 | Train Acc (20-way): 94.77%
Epoch 03 | Train Loss: 0.1306 | Train Acc (20-way): 95.75%
Epoch 04 | Train Loss: 0.1142 | Train Acc (20-way): 96.29%
Epoch 05 | Train Loss: 0.1026 | Train Acc (20-way): 96.78%
Epoch 06 | Train Loss: 0.0719 | Train Acc (20-way): 97.55%
Epoch 07 | Train Loss: 0.0788 | Train Acc (20-way): 97.26%
Epoch 08 | Train Loss: 0.0671 | Train Acc (20-way): 97.88%
Epoch 09 | Train Loss: 0.0591 | Train Acc (20-way): 97.98%
Epoch 10 | Train Loss: 0.0599 | Train Acc (20-way): 97.98%

Test Accuracy on Unseen Classes (5-way 5-shot): 98.60%
